### COPY INTO and Merge Commands
- Incrementaly loads data into Delta Lake Tables from Cloud Storage
- Supports schema evolution
- Supports wide range of file formats (csv, JSON, Parquet, Delta)
- Alternative to Auto Loader For Batch Ingestion
- Documentation: https://docs.databricks.com/aws/en/sql/language-manual/delta-copy-into

#### Create the  table to copy the data into

In [0]:
%sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;


In [0]:
%sql
DESC HISTORY demo.delta_lake.raw_stock_prices

In [0]:
%sql

DELETE FROM demo.delta_lake.raw_stock_prices;   --To remove duplicate scenario

COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@databrickscourseextmandl.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema'= 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
SELECT * FROM demo.delta_lake.raw_stock_prices

#### MERGE STATEMENT
- Used for upserts (Insert/Update/Delete oprations in a single statement)
- Allows merging new data into a target table based on matching condiation
- Documentation : https://docs.databricks.com/aws/en/sql/language-manual/delta-merge-into

In [0]:
%sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(
    stock_id STRING,
    price DOUBLE,
    trading_date DATE
);

1. Insert new stock received
2. update price and trading_date if updates received
3. Delete stocks which are delisted from the exchange(status="DELISTED")

In [0]:
%sql


MERGE INTO demo.delta_lake.stock_prices AS target
USING demo.delta_lake.raw_stock_prices AS source
    ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status ='ACTIVE' THEN 
    UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status ='DELISTED' THEN
    DELETE
WHEN NOT MATCHED AND source.status='ACTIVE' THEN 
    INSERT (stock_id, price, trading_date) VALUES (source.stock_id, source.price, source.trading_date);
    


In [0]:
%sql
SELECT * FROM demo.delta_lake.stock_prices